In [1]:
import numpy as np
from scipy.optimize import minimize

In [2]:
item_prices = np.load("HW_data/item_prices.npy")
print("item_prices size: ", np.size(item_prices))
print("item_prices first element: ", item_prices[0])

item_prices size:  100
item_prices first element:  19


In [3]:
assortments = np.load("HW_data/assortments.npy")
print("assortments size: ", np.shape(assortments))
print("assortments first element: ", assortments[0])
assortment_size = np.shape(assortments)[1]

assortments size:  (1000, 10)
assortments first element:  [72 32 29 63 34 66 82 20 85 30]


In [4]:
purchased_items = np.load("HW_data/purchased_items.npy")
print("purchased_items size: ", np.size(purchased_items))
print("purchased_items first element: ", purchased_items[0])

purchased_items size:  1000
purchased_items first element:  85


In [5]:
item_features = np.load("HW_data/item_features.npy")
print("item features size: ", np.shape(item_features))
print("item_features first element: ", item_features[0])

item features size:  (100, 10)
item_features first element:  [19.          0.35181949  0.68242623  0.41669946  0.22766009  0.79645528
  0.03878769  0.81046195  0.90673804  0.69007512]


In [25]:
#initialize params
beta = np.zeros(10)
np.shape(beta)

(10,)

In [26]:
#purchased items are 1 based indexed, but python is 
def neg_log_likelihood(beta, assortments, purchased_items, item_features):
    tmp_sum = 0
    for t in range(np.size(purchased_items)):
        if purchased_items[t] > 0:
            tmp_sum += np.sum(item_features[purchased_items[t]-1] * beta)
        tmp_sum -= np.log(np.sum(np.exp(np.sum(item_features[assortments[t]-1] * beta, axis=1))) + 1)
    return -tmp_sum
neg_log_likelihood(beta, assortments, purchased_items, item_features)

2397.895272798411

In [10]:
opt_params = minimize(neg_log_likelihood, beta, args=(assortments, purchased_items, item_features))

In [21]:
estimated_beta = opt_params.x

In [22]:
def convert_beta_to_params(beta, item_features):
    return np.exp(np.sum(item_features*beta, axis = 1))
params = convert_beta_to_params(estimated_beta, item_features)

In [23]:
#black box assortment optimization
#inputs parameters, prices
#output optimal assortment
def find_optimal_assortment(params, prices):
    best_assortment = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    best_revenue = 0
    for i in np.arange(150,250)/10:
        #calculate v_i * (r_i - r*) & pick 10 items with the highest value
        values = np.sort((prices - i) * params)[-10:]
        assortment = []
        for j in values:
            index = np.where((prices - i) * params == j)
            assortment.append(index[0][0])

        probs = params[assortment] / (sum(params[assortment]) + 1)
        revenue = sum(probs * prices[assortment])
        if revenue > best_revenue:
            best_revenue = revenue
            best_assortment = assortment
    #shift assortments from 0 based indexing to 1 based
    best_assortment = (best_assortment + np.ones(10)).astype(int)
    return best_assortment, best_revenue

In [24]:
estimated_optimal_assortment, estimated_optimal_revenue = find_optimal_assortment(params, item_prices)
estimated_optimal_assortment, estimated_optimal_revenue

(array([75, 25, 73, 38, 59, 93, 65, 16, 37, 77]), 22.789127417888086)